# LLMs and Tool Calling: From Text to Action

This notebook is structured in three parts:

1. **What LLMs Can Do** — Practical applications where LLMs shine on their own
2. **Where LLMs Fail** — Three clear demonstrations of LLM limitations
3. **Tool Calling as the Solution** — How tools fix each limitation, with live examples

The goal is not just to show code, but to understand *why* each pattern exists and what problem it solves.

---

## Setup: Install Dependencies

Run this cell once to install everything this notebook needs.

In [ ]:
!uv pip install langchain langchain-groq langchain-core pydantic python-dotenv requests -q

## Configuration

We use **Groq** as our LLM provider because it is fast and free to get started.

Get your API key at: https://console.groq.com

Then either:
- Create a `.env` file with `GROQ_API_KEY=your_key_here`, or
- Paste your key directly below

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

MODEL_NAME   = "llama-3.1-8b-instant"
TEMPERATURE  = 0.7

print("Configuration loaded.")
print(f"Model: {MODEL_NAME}")

Configuration loaded.
Model: llama-3.1-8b-instant


---

# Part 1: What LLMs Can Do

Before we explore limitations, let us appreciate what LLMs do exceptionally well.

An LLM is a pattern-completion engine trained on massive amounts of text. Any task where the answer is **contained within the language itself** — summarizing, translating, explaining, classifying — is a natural fit.

Think of it like this: an LLM is a brilliant scholar locked in a library. Give it a problem that can be solved with the books on its shelves, and it will excel. Problems that require stepping outside the library? That is where it struggles.

---

## Let's consider how LLM can be used for summarization

A classic and genuinely useful LLM application. Given a long piece of text, the LLM extracts the key ideas and presents them concisely.

We use **Pydantic** to enforce structured output — instead of a raw string, we get back a clean Python object with a `title` and `summary` field.

In [3]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field


# --- Define the output structure ---
class SummaryOutput(BaseModel):
    title:   str = Field(..., description="A catchy title for the summary.")
    summary: str = Field(..., description="A concise summary of the provided text.")


# --- Build the chain ---
def summarize_text(text: str) -> SummaryOutput:
    llm = ChatGroq(api_key=GROQ_API_KEY, model=MODEL_NAME, temperature=TEMPERATURE)
    parser = PydanticOutputParser(pydantic_object=SummaryOutput)

    prompt = PromptTemplate(
        template=(
            "You are an expert text summarizer.\n"
            "Given the following text, provide a concise summary and a catchy title.\n\n"
            "Text:\n{text}\n\n"
            "Output format:\n{format_instructions}"
        ),
        input_variables=["text"],
        partial_variables={"format_instructions": parser.get_format_instructions()},
    )

    chain = prompt | llm | parser
    return chain.invoke({"text": text})


# --- Run it ---
sample_text = """
Artificial Intelligence (AI) has rapidly evolved over the past decade, transforming 
various industries and aspects of daily life. From healthcare to finance, AI technologies 
are being leveraged to improve efficiency, accuracy, and decision-making processes. 
Machine learning algorithms, natural language processing, and computer vision are some 
of the key areas driving this revolution. As AI continues to advance, ethical 
considerations and regulatory frameworks are becoming increasingly important to ensure 
responsible development and deployment of these technologies.
"""

result = summarize_text(sample_text)

print("=" * 60)
print(f"TITLE:   {result.title}")
print("-" * 60)
print(f"SUMMARY: {result.summary}")
print("=" * 60)

TITLE:   AI Revolution and the Need for Responsible Development
------------------------------------------------------------
SUMMARY: Artificial Intelligence has rapidly transformed industries and daily life, improving efficiency and decision-making. As AI advances, ethical considerations and regulatory frameworks become increasingly important.


**Why did this work?**

Because everything the LLM needed was *inside the prompt itself*. The text was provided, and summarization is a task the LLM has learned from billions of examples. No external data was required.

### Question: Do you know some other tasks where LLMS work pretty well?


Now let us see what happens when that condition breaks.

---

# Part 2: Where LLMs Fail

The LLM's fundamental constraint is this: **it is frozen in time, isolated from the world, and cannot act**.

It was trained on data up to a certain date, has no connection to external systems, and cannot do anything except return text.

The following three cells demonstrate each failure clearly and deliberately.

---

## Failure 1: LLM Does Not Know Your Private Data

The LLM has never seen your company's internal documents, databases, or reports. If you ask it about them, it either says it does not know, or worse — it makes something up (hallucination).

In [4]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=GROQ_API_KEY, model=MODEL_NAME, temperature=0)

question = "What was Lumivya Technology's total revenue in Q3 2025?"

response = llm.invoke(question)

print("QUESTION:", question)
print()
print("LLM RESPONSE:")
print(response.content)
print()
print("[!] Notice: The LLM either refuses or guesses. Neither is useful.")

QUESTION: What was Lumivya Technology's total revenue in Q3 2025?

LLM RESPONSE:
I cannot verify Lumivya Technology's total revenue in Q3 2025.

[!] Notice: The LLM either refuses or guesses. Neither is useful.


---

## Failure 2: LLM Cannot Access Live Data

The LLM's training has a cutoff date. Any question about the current state of the world — prices, news, live statistics — will return stale or incorrect information.

The LLM does not know it is wrong. It answers with the same confidence regardless of whether the information is current or outdated.

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=GROQ_API_KEY, model=MODEL_NAME, temperature=0)

question = "What is the current price of Bitcoin in USD right now?"

response = llm.invoke(question)

print("QUESTION:", question)
print()
print("LLM RESPONSE:")
print(response.content)
print()
print("[!] The LLM gives a price from its training data — which could be years old.")
print("[!] It cannot access the internet. It does not even know today's date.")

QUESTION: What is the current price of Bitcoin in USD right now?

LLM RESPONSE:
I can't provide real-time information or current prices. However, I can suggest some reliable sources where you can find the current price of Bitcoin in USD:

1. **CoinMarketCap**: A popular website that provides real-time cryptocurrency prices, including Bitcoin.
2. **CoinGecko**: Another reliable source for cryptocurrency prices, including Bitcoin.
3. **Google Finance**: You can search for "Bitcoin price" on Google Finance to get the current price.
4. **Yahoo Finance**: Similar to Google Finance, you can search for "Bitcoin price" on Yahoo Finance.

Please note that cryptocurrency prices can fluctuate rapidly, so it's always a good idea to check multiple sources for the most up-to-date information.

[!] The LLM gives a price from its training data — which could be years old.
[!] It cannot access the internet. It does not even know today's date.


---

## Failure 3: LLM Cannot Take Real-World Actions

Ask an LLM to save a file, send an email, or update a database record. It will write beautiful text *describing* how it would do that. Nothing will actually happen.

The LLM is text-in, text-out. By itself, it has no hands.

In [6]:
import os
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=GROQ_API_KEY, model=MODEL_NAME, temperature=0)

question = "Save this note to a file called notes.txt: 'Team meeting at 3pm today.'"

response = llm.invoke(question)

print("QUESTION:", question)
print()
print("LLM RESPONSE:")
print(response.content)
print()

# Check if the file was actually created
if os.path.exists("notes.txt"):
    print("[+] notes.txt exists.")
else:
    print("[!] notes.txt does NOT exist. The LLM described what to do but did nothing.")
    print("[!] Words are not actions. The file was never created.")

QUESTION: Save this note to a file called notes.txt: 'Team meeting at 3pm today.'

LLM RESPONSE:
I can simulate saving a note to a file. However, I'm a text-based AI and do not have the capability to directly interact with your file system. But I can provide you with the command to save the note to a file called notes.txt:

**For Windows:**

1. Open Notepad or any text editor of your choice.
2. Type the note: 'Team meeting at 3pm today.'
3. Click on "File" > "Save As" and select the location where you want to save the file.
4. Name the file "notes.txt" and select "All Files" as the file type.
5. Click "Save".

**For macOS or Linux:**

1. Open a text editor like TextEdit or Gedit.
2. Type the note: 'Team meeting at 3pm today.'
3. Go to "File" > "Save As" and select the location where you want to save the file.
4. Name the file "notes.txt" and select "Plain Text" as the file type.
5. Click "Save".

Alternatively, you can use the command line to save the note to a file. Here's how:

**For

---

# Part 3: Tool Calling — The Solution

Tool calling gives the LLM a set of **callable functions**. The LLM can now:
1. Recognize when it needs a tool
2. Select the right tool
3. Send it a structured request
4. Use the result to form a final answer

The LLM is still the brain — it decides what to do and how to communicate. But now it has hands.

We use `verbose=True` throughout so you can watch the agent's reasoning steps live. That inner monologue is the most important thing to observe.

---

## Solution 1: Connecting the LLM to Private Data

We define a `get_company_revenue` tool that simulates a private company database. The LLM now knows it can call this tool instead of guessing.

In [12]:
import langchain
print(langchain.__version__)

1.2.13


In [12]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage, ToolMessage


# --- Simulated private company database ---
COMPANY_DATA = {
    "Q1 2024": "$1.2M",
    "Q2 2024": "$1.5M",
    "Q3 2024": "$1.8M",
    "Q4 2024": "$2.1M",
}


@tool
def get_company_revenue(quarter: str) -> str:
    """
    Fetch quarterly revenue data from the company's internal database.
    Use this tool when asked about company revenue or financial performance.
    Input should be a quarter string like 'Q3 2024'.
    """
    result = COMPANY_DATA.get(quarter)
    if result:
        return f"Revenue for {quarter}: {result}"
    return f"No data found for '{quarter}'. Available quarters: {list(COMPANY_DATA.keys())}"


# --- Build the agent ---
llm   = ChatGroq(api_key=GROQ_API_KEY, model=MODEL_NAME, temperature=0)
tools = [get_company_revenue]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful business assistant. Always use the available tools to answer questions about company data.",
)

print("=" * 60)
print("QUESTION: What was our company revenue in Q3 2024?")
print("=" * 60)

response = agent.invoke({"messages": [{"role": "user", "content": "What was our company revenue in Q3 2024?"}]})

print()
messages = response["messages"]

# Get all AI messages
ai_messages = [m for m in messages if isinstance(m, AIMessage)]

# Get all tool messages
tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

# Get the final AI response (last AI message)
final_answer = next(m for m in reversed(messages) if isinstance(m, AIMessage))

print("TOOL RESPONSES:")
for tool_message in tool_messages:
    print(tool_message.content)

print("----------")
print("FINAL LLM ANSWER:", final_answer.content)

/tmp/ipykernel_19654/3873760868.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


QUESTION: What was our company revenue in Q3 2024?

TOOL RESPONSES:
Revenue for Q3 2024: $1.8M
----------
FINAL LLM ANSWER: Note: The actual revenue value is not provided here as it is a hypothetical scenario. The value $1.8M is just a placeholder.


**What just happened?**

Read the verbose output above carefully. You will see the agent:
1. Recognized the question requires company data it doesn't have
2. Decided to call `get_company_revenue` with `quarter = "Q3 2024"`
3. Received `$1.8M` back from our simulated database
4. Formed a natural language answer using that real data

The LLM did not guess. It asked the right tool the right question.

---

## Solution 2: Accessing Live Real-World Data

We give the LLM a tool that calls a live external API. The CoinGecko API is free and requires no API key — perfect for a live demonstration.

Watch the difference from before: this time, the price is real and current.

In [13]:
import requests
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage, ToolMessage


@tool
def get_crypto_price(coin_id: str) -> str:
    """
    Fetch the current price of a cryptocurrency in USD using the CoinGecko API.
    Use this for any question about current crypto prices.
    Input should be a coin id like 'bitcoin', 'ethereum', or 'solana'.
    """
    try:
        response = requests.get(
            "https://api.coingecko.com/api/v3/simple/price",
            params={"ids": coin_id, "vs_currencies": "usd"},
            timeout=10,
        )
        data = response.json()
        if coin_id in data:
            price = data[coin_id]["usd"]
            return f"Current price of {coin_id}: ${price:,.2f} USD (live data)"
        return f"Could not find price data for '{coin_id}'."
    except Exception as e:
        return f"Error fetching price: {str(e)}"


# --- Build the agent ---
llm   = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)  # upgraded model
tools = [get_crypto_price]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful financial assistant. Always use the get_crypto_price tool to get current data. Do not use any other tools.",
)

print("=" * 60)
print("QUESTION: What is the current price of Bitcoin and Ethereum?")
print("=" * 60)

response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the current price of Bitcoin and Ethereum?"}]
})

print()
messages = response["messages"]

# Get all AI messages
ai_messages = [m for m in messages if isinstance(m, AIMessage)]

# Get all tool messages
tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

# Get the final AI response (last AI message)
final_answer = next(m for m in reversed(messages) if isinstance(m, AIMessage))

print("TOOL RESPONSES:")
for tool_message in tool_messages:
    print(tool_message.content)

print("----------")
print("FINAL LLM ANSWER:", final_answer.content)

/tmp/ipykernel_19654/595673487.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


QUESTION: What is the current price of Bitcoin and Ethereum?

TOOL RESPONSES:
Current price of bitcoin: $66,108.00 USD (live data)
Current price of ethereum: $1,992.61 USD (live data)
----------
FINAL LLM ANSWER: The current price of Bitcoin is $66,108.00 USD and the current price of Ethereum is $1,992.61 USD.


**Notice:** The agent called the tool twice — once for Bitcoin, once for Ethereum. It figured out on its own that two tool calls were needed. That decision-making is the agent at work.

---

## Solution 3: Taking Real-World Actions

Now the LLM can actually *do* things — not just describe them. We give it tools to save and read notes from a real file on disk.

After running this cell, open `notes.txt` in your file explorer. The file will actually exist.

In [14]:
import os
from datetime import datetime
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage, ToolMessage

NOTES_FILE = "notes.txt"


@tool
def save_note(note: str) -> str:
    """
    Save a note to the notes.txt file with a timestamp.
    Use this when the user wants to save, write, or record something.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry     = f"[{timestamp}] {note}\n"

    with open(NOTES_FILE, "a") as f:
        f.write(entry)

    return f"Note saved successfully: '{note}'"


@tool
def read_notes() -> str:
    """
    Read all notes from the notes.txt file.
    Use this when the user asks what notes they have or wants to review their notes.
    """
    if not os.path.exists(NOTES_FILE):
        return "No notes found. The notes file does not exist yet."

    with open(NOTES_FILE, "r") as f:
        contents = f.read().strip()

    return f"Your saved notes:\n{contents}" if contents else "The notes file is empty."


# --- Build the agent ---
llm   = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)
tools = [save_note, read_notes]

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a helpful personal assistant. Use tools to save and retrieve notes.",
)

# --- Action 1: Save a note ---
print("=" * 60)
print("ACTION 1: Saving a note")
print("=" * 60)

agent.invoke({"messages": [{"role": "user", "content": "Save a note: Team meeting at 3pm today."}]})
agent.invoke({"messages": [{"role": "user", "content": "Also save this: Follow up with the design team about the new logo."}]})

# --- Action 2: Read notes back ---
print()
print("=" * 60)
print("ACTION 2: Reading notes back")
print("=" * 60)

response = agent.invoke({"messages": [{"role": "user", "content": "What notes do I have saved?"}]})

messages     = response["messages"]
tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
final_answer  = next(m for m in reversed(messages) if isinstance(m, AIMessage))

print()
print("TOOL RESPONSES:")
for tool_message in tool_messages:
    print(tool_message.content)

print()
print("FINAL ANSWER:", final_answer.content)
print()
print(f"[+] Check your directory — '{NOTES_FILE}' now physically exists on disk.")

/tmp/ipykernel_19654/2326418239.py:45: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


ACTION 1: Saving a note

ACTION 2: Reading notes back

TOOL RESPONSES:
Your saved notes:
[2026-03-26 13:55:57] Team meeting at 3pm today.
[2026-03-26 13:55:58] Follow up with the design team about the new logo.
[2026-03-26 13:57:20] Bitcoin price: [insert price], Q3 2024 company revenue: [insert revenue]
[2026-03-28 16:27:29] Team meeting at 3pm today.
[2026-03-28 16:27:30] Follow up with the design team about the new logo.

FINAL ANSWER: You have the following notes saved:
- Team meeting at 3pm today (from March 26 and March 28)
- Follow up with the design team about the new logo (from March 26 and March 28)
- Bitcoin price and Q3 2024 company revenue (from March 26)

[+] Check your directory — 'notes.txt' now physically exists on disk.


---

## Tool Calling with Multiple Tools (Chaining)

In real systems, an LLM agent does not just call one tool — it chains multiple tools together to complete a complex task.

This final example shows an agent that: fetches live crypto data, stores the finding as a note, and then summarizes everything — all from a single instruction.

In [15]:
import requests
import os
from datetime import datetime
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage, ToolMessage


@tool
def get_crypto_price(coin_id: str) -> str:
    """
    Fetch the current price of a cryptocurrency in USD.
    Input should be a coin id like 'bitcoin' or 'ethereum'.
    """
    try:
        response = requests.get(
            "https://api.coingecko.com/api/v3/simple/price",
            params={"ids": coin_id, "vs_currencies": "usd"},
            timeout=10,
        )
        data  = response.json()
        price = data[coin_id]["usd"]
        return f"{coin_id} current price: ${price:,.2f} USD"
    except Exception as e:
        return f"Error: {str(e)}"


@tool
def save_note(note: str) -> str:
    """
    Save a note with a timestamp to summary.txt.
    Use this to record important findings or information.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("summary.txt", "a") as f:
        f.write(f"[{timestamp}] {note}\n")
    return f"Saved: '{note}'"


@tool
def get_company_revenue(quarter: str) -> str:
    """
    Get company revenue for a given quarter from internal records.
    Input should be like 'Q3 2024'.
    """
    data = {"Q1 2024": "$1.2M", "Q2 2024": "$1.5M", "Q3 2024": "$1.8M", "Q4 2024": "$2.1M"}
    return data.get(quarter, f"No data for {quarter}")


# --- Build agent with ALL tools ---
llm   = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0)
tools = [get_crypto_price, save_note, get_company_revenue]


# Using basic prompt: The summary stored is not properly using the actual values from the tools. The LLM is just writing placeholder text instead of the real price and revenue.
# agent = create_react_agent(
#     model=llm,
#     tools=tools,
#     prompt=(
#         "You are a smart business assistant. You have tools to fetch live data, "
#         "check company records, and save notes. Use them in combination to complete tasks."
#     ),
# )

# # --- A complex multi-tool task ---
# task = (
#     "Do the following: "
#     "1) Get the current Bitcoin price. "
#     "2) Get our Q3 2024 company revenue. "
#     "3) Save a note summarizing both findings. "
#     "4) Give me a final summary."
# )

# Using better prompt
agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=(
        "You are a smart business assistant. You have tools to fetch live data, "
        "check company records, and save notes. Use them in combination to complete tasks. "
        "IMPORTANT: When saving notes, always use the ACTUAL values returned by the tools. "
        "Never use placeholder text like [insert price] or [insert value]."
    ),
)

# --- A complex multi-tool task ---
task = (
    "Do the following steps in order: "
    "1) Call get_crypto_price('bitcoin') and remember the exact price returned. "
    "2) Call get_company_revenue('Q3 2024') and remember the exact revenue returned. "
    "3) Call save_note with a summary that includes the ACTUAL price and revenue values from steps 1 and 2. "
    "4) Give me a final summary of what you found."
)

print("=" * 60)
print("TASK:", task)
print("=" * 60)

response = agent.invoke({"messages": [{"role": "user", "content": task}]})

messages      = response["messages"]
tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
final_answer  = next(m for m in reversed(messages) if isinstance(m, AIMessage))

print()
print("TOOL RESPONSES:")
for tool_message in tool_messages:
    print(f"  [{tool_message.name}]: {tool_message.content}")

print()
print("FINAL ANSWER:")
print(final_answer.content)

/tmp/ipykernel_19654/565754128.py:76: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


TASK: Do the following steps in order: 1) Call get_crypto_price('bitcoin') and remember the exact price returned. 2) Call get_company_revenue('Q3 2024') and remember the exact revenue returned. 3) Call save_note with a summary that includes the ACTUAL price and revenue values from steps 1 and 2. 4) Give me a final summary of what you found.

TOOL RESPONSES:
  [get_crypto_price]: bitcoin current price: $66,128.00 USD
  [get_company_revenue]: $1.8M
  [save_note]: Saved: 'Bitcoin price: $66128.00, Company revenue for Q3 2024: $1,800,000'

FINAL ANSWER:
The current price of bitcoin is $66,128.00 USD, and the company revenue for Q3 2024 is $1,800,000. A note with this information has been saved to the summary.txt file.


## Closing Notes
- This is the transition from language model to autonomous system. And what we are building in this bootcamp An AI Data Analyst Agent is exactly this. A brain with tools, memory, and the ability to act.


### A Question: "If an LLM decides which tool to call, based on a docstring it reads -- who is really making the decision? The programmer who wrote the tool, or the model?"